In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import os
import joblib as jb
from datetime import datetime
import src.raw_preprocessing as rp

## Conso  
Source : https://www.rte-france.com/donnees-publications/eco2mix-donnees-temps-reel/telecharger-indicateurs

In [4]:
conso = pd.DataFrame() 
for i in range(1, 5):
    df = pd.read_csv(f"data/conso/conso_energie_202{i}.xls", sep="\t", encoding="latin1", low_memory=False)
    conso = pd.concat([conso,df])


In [3]:
conso.columns

Index(['Périmètre', 'Nature', 'Date', 'Heures', 'Consommation',
       'Prévision J-1', 'Prévision J', 'Fioul', 'Charbon', 'Gaz', 'Nucléaire',
       'Eolien', 'Solaire', 'Hydraulique', 'Pompage', 'Bioénergies',
       'Ech. physiques', 'Taux de Co2', 'Ech. comm. Angleterre',
       'Ech. comm. Espagne', 'Ech. comm. Italie', 'Ech. comm. Suisse',
       'Ech. comm. Allemagne-Belgique', 'Fioul - TAC', 'Fioul - Cogén.',
       'Fioul - Autres', 'Gaz - TAC', 'Gaz - Cogén.', 'Gaz - CCG',
       'Gaz - Autres', 'Hydraulique - Fil de l?eau + éclusée',
       'Hydraulique - Lacs', 'Hydraulique - STEP turbinage',
       'Bioénergies - Déchets', 'Bioénergies - Biomasse',
       'Bioénergies - Biogaz', ' Stockage batterie', 'Déstockage batterie',
       'Eolien terrestre', 'Eolien offshore'],
      dtype='str')

In [4]:
conso = conso.shift(axis=1)
conso["Périmètre"] = conso.index
conso.head(1)

,Périmètre,Nature,Date,Heures,Consommation,Prévision J-1,Prévision J,Fioul,Charbon,Gaz,...,Hydraulique - Fil de l?eau + éclusée,Hydraulique - Lacs,Hydraulique - STEP turbinage,Bioénergies - Déchets,Bioénergies - Biomasse,Bioénergies - Biogaz,Stockage batterie,Déstockage batterie,Eolien terrestre,Eolien offshore
France,France,Données définitives,2021-01-01,00:00,67156.0,67000.0,66800.0,94.0,8.0,5585.0,...,5163.0,1371.0,988.0,502.0,288.0,335.0,ND,ND,ND,ND


In [5]:
conso = conso.reset_index()
conso = conso.drop(["index"], axis=1)
conso.head(1)

,Périmètre,Nature,Date,Heures,Consommation,Prévision J-1,Prévision J,Fioul,Charbon,Gaz,...,Hydraulique - Fil de l?eau + éclusée,Hydraulique - Lacs,Hydraulique - STEP turbinage,Bioénergies - Déchets,Bioénergies - Biomasse,Bioénergies - Biogaz,Stockage batterie,Déstockage batterie,Eolien terrestre,Eolien offshore
0,France,Données définitives,2021-01-01,00:00,67156.0,67000.0,66800.0,94.0,8.0,5585.0,...,5163.0,1371.0,988.0,502.0,288.0,335.0,ND,ND,ND,ND


In [6]:
conso.head()

,Périmètre,Nature,Date,Heures,Consommation,Prévision J-1,Prévision J,Fioul,Charbon,Gaz,...,Hydraulique - Fil de l?eau + éclusée,Hydraulique - Lacs,Hydraulique - STEP turbinage,Bioénergies - Déchets,Bioénergies - Biomasse,Bioénergies - Biogaz,Stockage batterie,Déstockage batterie,Eolien terrestre,Eolien offshore
0,France,Données définitives,2021-01-01,00:00,67156.0,67000.0,66800.0,94.0,8.0,5585.0,...,5163.0,1371.0,988.0,502.0,288.0,335.0,ND,ND,ND,ND
1,France,Données définitives,2021-01-01,00:15,NaN,66500.0,66300.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,France,Données définitives,2021-01-01,00:30,66483.0,66000.0,65800.0,88.0,7.0,5599.0,...,4991.0,1280.0,1029.0,511.0,297.0,332.0,0,0,0,0
3,France,Données définitives,2021-01-01,00:45,NaN,65150.0,64900.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,France,Données définitives,2021-01-01,01:00,64390.0,64300.0,64000.0,89.0,7.0,4476.0,...,4891.0,829.0,476.0,511.0,291.0,332.0,0,0,0,0


There is a missing value in `Consommation` every 15 min, by starting at 00:00

In [7]:
conso[conso["Consommation"].isnull()]

,Périmètre,Nature,Date,Heures,Consommation,Prévision J-1,Prévision J,Fioul,Charbon,Gaz,...,Hydraulique - Fil de l?eau + éclusée,Hydraulique - Lacs,Hydraulique - STEP turbinage,Bioénergies - Déchets,Bioénergies - Biomasse,Bioénergies - Biogaz,Stockage batterie,Déstockage batterie,Eolien terrestre,Eolien offshore
1,France,Données définitives,2021-01-01,00:15,NaN,66500.0,66300.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,France,Données définitives,2021-01-01,00:45,NaN,65150.0,64900.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,France,Données définitives,2021-01-01,01:15,NaN,64600.0,64350.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,France,Données définitives,2021-01-01,01:45,NaN,64600.0,64550.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,France,Données définitives,2021-01-01,02:15,NaN,63700.0,63450.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140252,France,Données définitives,2024-12-31,22:15,NaN,61250.0,61500.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140254,France,Données définitives,2024-12-31,22:45,NaN,62950.0,63100.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140256,France,Données définitives,2024-12-31,23:15,NaN,64500.0,64550.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140258,France,Données définitives,2024-12-31,23:45,NaN,64400.0,64500.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We can observe that the energy consumption numbers are listed every 30 minutes.  
So we'll remove the other rows.

In [8]:
conso = conso[conso["Consommation"].notna()]
conso.info()

<class 'pandas.DataFrame'>
Index: 70128 entries, 0 to 140257
Data columns (total 40 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Périmètre                             70128 non-null  str    
 1   Nature                                70128 non-null  str    
 2   Date                                  70128 non-null  str    
 3   Heures                                70128 non-null  str    
 4   Consommation                          70128 non-null  float64
 5   Prévision J-1                         70128 non-null  float64
 6   Prévision J                           70128 non-null  float64
 7   Fioul                                 70128 non-null  float64
 8   Charbon                               70128 non-null  float64
 9   Gaz                                   70128 non-null  float64
 10  Nucléaire                             70128 non-null  float64
 11  Eolien                        

In [9]:
conso = conso[["Date", "Heures", "Consommation"]]
conso

,Date,Heures,Consommation
0,2021-01-01,00:00,67156.0
2,2021-01-01,00:30,66483.0
4,2021-01-01,01:00,64390.0
6,2021-01-01,01:30,64223.0
8,2021-01-01,02:00,63689.0
...,...,...,...
140249,2024-12-31,21:30,63946.0
140251,2024-12-31,22:00,63063.0
140253,2024-12-31,22:30,63816.0
140255,2024-12-31,23:00,65437.0


## Conso (py pipeline)

In [12]:
conso2 = rp.conso_preprocess("data/conso/")
conso2

,Date,Heures,Consommation
0,2021-01-01,00:00,67156.0
2,2021-01-01,00:30,66483.0
4,2021-01-01,01:00,64390.0
6,2021-01-01,01:30,64223.0
8,2021-01-01,02:00,63689.0
...,...,...,...
140249,2024-12-31,21:30,63946.0
140251,2024-12-31,22:00,63063.0
140253,2024-12-31,22:30,63816.0
140255,2024-12-31,23:00,65437.0


## School holidays  
Source : https://www.data.gouv.fr/datasets/le-calendrier-scolaire

In [10]:
holidays = pd.read_excel("data/calendar/calendrier_gouv.xlsx")

In [11]:
holidays["Zones"] = holidays["Zones"].apply(lambda d: d.replace(' ', '_'))
print(np.unique(holidays["Zones"]))
holidays

['Corse' 'Guadeloupe' 'Guyane' 'Martinique' 'Mayotte' 'Nouvelle_Calédonie'
 'Polynésie' 'Réunion' 'Saint_Pierre_et_Miquelon' 'Wallis_et_Futuna'
 'Zone_A' 'Zone_B' 'Zone_C']


,Description,Population,Date de début,Date de fin,Académies,Zones,annee_scolaire
0,Vacances de la Toussaint,-,2017-10-20T22:00:00+00:00,2017-11-05T23:00:00+00:00,Besançon,Zone_A,2017-2018
1,Vacances de la Toussaint,-,2017-10-20T22:00:00+00:00,2017-11-05T23:00:00+00:00,Dijon,Zone_A,2017-2018
2,Vacances de Noël,-,2017-12-22T23:00:00+00:00,2018-01-07T23:00:00+00:00,Lyon,Zone_A,2017-2018
3,Vacances d'Hiver,-,2018-02-09T23:00:00+00:00,2018-02-25T23:00:00+00:00,Dijon,Zone_A,2017-2018
4,Vacances d'Hiver,-,2018-02-09T23:00:00+00:00,2018-02-25T23:00:00+00:00,Poitiers,Zone_A,2017-2018
...,...,...,...,...,...,...,...
2315,Vacances de Printemps,-,2021-03-31T22:00:00+00:00,2021-04-18T22:00:00+00:00,Guyane,Guyane,2020-2021
2316,Vacances de Carnaval,-,2024-02-09T23:00:00+00:00,2024-02-25T23:00:00+00:00,Guyane,Guyane,2023-2024
2317,Pont à Pâques,-,2024-03-27T23:00:00+00:00,2024-04-02T22:00:00+00:00,Guyane,Guyane,2023-2024
2318,Pont de l'Ascension,-,2024-05-08T22:00:00+00:00,2024-05-12T22:00:00+00:00,Guyane,Guyane,2023-2024


In [12]:
holidays[
    (holidays["Zones"] == "Zone_A") & 
    (holidays["Description"] == "Vacances de Noël") &
    (holidays["annee_scolaire"] == "2020-2021")
    ].head()

,Description,Population,Date de début,Date de fin,Académies,Zones,annee_scolaire
38,Vacances de Noël,-,2020-12-18T23:00:00+00:00,2021-01-03T23:00:00+00:00,Bordeaux,Zone_A,2020-2021
39,Vacances de Noël,-,2020-12-18T23:00:00+00:00,2021-01-03T23:00:00+00:00,Dijon,Zone_A,2020-2021
482,Vacances de Noël,-,2020-12-18T23:00:00+00:00,2021-01-03T23:00:00+00:00,Besançon,Zone_A,2020-2021
483,Vacances de Noël,-,2020-12-18T23:00:00+00:00,2021-01-03T23:00:00+00:00,Grenoble,Zone_A,2020-2021
484,Vacances de Noël,-,2020-12-18T23:00:00+00:00,2021-01-03T23:00:00+00:00,Lyon,Zone_A,2020-2021


In [13]:
holidays["Description"].unique()

<StringArray>
[          'Vacances de la Toussaint',                   'Vacances de Noël',
                   'Vacances d'Hiver',              'Vacances de Printemps',
                     'Vacances d'Été',                'Pont de l'Ascension',
           'Début des Vacances d'Été',           'Abolition de l’esclavage',
                     'Semaine en mai',                          'Mi-carême',
               'Vacances de Carnaval',                               'Pont',
                 'Vacances de Pâques',        'Vacances après 1ère période',
           'Vacances d'Hiver austral',        'Vacances après 3ème période',
        'Vacances après 4ème période',                   'Grandes Vacances',
                    'Vacances de Mai',              'Vacances de Septembre',
                'Vacances de Février',              'Vacances 1ère période',
              'Vacances 4ème période',              'Vacances 3ème période',
              'Vacances 2ème période', 'Début des Vacances d'H

We'll focus on `{"Vacances de la Toussaint", "Vacances de Noël", "Vacances d'Hiver", "Vacances de Printemps", "Vacances d'Été"}`

In [14]:
holiday_names = [
    "Vacances de la Toussaint", 
    "Vacances de Noël", 
    "Vacances d'Hiver", 
    "Vacances de Printemps", 
    "Vacances d'Été"
]
zones = ["Zone_A", "Zone_B", "Zone_C"]



new_holidays = conso[["Date"]]

for zone in zones:    
    new_holidays[zone] = False

for name in holiday_names:
    new_holidays[name] = 0

In [15]:
new_holidays

,Date,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,False,False,False,0,0,0,0,0
2,2021-01-01,False,False,False,0,0,0,0,0
4,2021-01-01,False,False,False,0,0,0,0,0
6,2021-01-01,False,False,False,0,0,0,0,0
8,2021-01-01,False,False,False,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,False,False,False,0,0,0,0,0
140251,2024-12-31,False,False,False,0,0,0,0,0
140253,2024-12-31,False,False,False,0,0,0,0,0
140255,2024-12-31,False,False,False,0,0,0,0,0


#### We add informations about the holidays

In [16]:
# We store the holidays for each zone in a dictionary
holiday_ranges = {}

for zone in zones:

    # We select the date of the new dataset to facilitate the automation of the project in the future
    min_year = int(new_holidays["Date"].iloc[0][:4])
    max_year = int(new_holidays["Date"].iloc[-1][:4])

    ranges = []
    for date in range(min_year, max_year+1):
        for holiday in holiday_names:
            # The holidays "Vacances de Noël" start at "date-1" and end at "date".
            if holiday == "Vacances de Noël":
                holidays_filtered = holidays[
                                    (holidays["Zones"] == zone) & 
                                    (holidays["Description"] == holiday) &
                                    (holidays["annee_scolaire"] == f"{date-1}-{date}")
                                ]
            else : 
                holidays_filtered = holidays[
                                        (holidays["Zones"] == zone) & 
                                        (holidays["Description"] == holiday) &
                                        (holidays["annee_scolaire"] == f"{date}-{date+1}")
                                    ]
    
            start = datetime.strptime(holidays_filtered["Date de début"].iloc[0][:10], "%Y-%m-%d")
            end = datetime.strptime(holidays_filtered["Date de fin"].iloc[0][:10], "%Y-%m-%d")
            ranges.append((holiday, start, end))
    
    holiday_ranges[zone] = ranges

holiday_ranges

{'Zone_A': [('Vacances de la Toussaint',
   datetime.datetime(2021, 10, 22, 0, 0),
   datetime.datetime(2021, 11, 7, 0, 0)),
  ('Vacances de Noël',
   datetime.datetime(2020, 12, 18, 0, 0),
   datetime.datetime(2021, 1, 3, 0, 0)),
  ("Vacances d'Hiver",
   datetime.datetime(2022, 2, 11, 0, 0),
   datetime.datetime(2022, 2, 27, 0, 0)),
  ('Vacances de Printemps',
   datetime.datetime(2022, 4, 15, 0, 0),
   datetime.datetime(2022, 5, 1, 0, 0)),
  ("Vacances d'Été",
   datetime.datetime(2022, 7, 6, 0, 0),
   datetime.datetime(2022, 8, 31, 0, 0)),
  ('Vacances de la Toussaint',
   datetime.datetime(2022, 10, 21, 0, 0),
   datetime.datetime(2022, 11, 6, 0, 0)),
  ('Vacances de Noël',
   datetime.datetime(2021, 12, 17, 0, 0),
   datetime.datetime(2022, 1, 2, 0, 0)),
  ("Vacances d'Hiver",
   datetime.datetime(2023, 2, 3, 0, 0),
   datetime.datetime(2023, 2, 19, 0, 0)),
  ('Vacances de Printemps',
   datetime.datetime(2023, 4, 7, 0, 0),
   datetime.datetime(2023, 4, 23, 0, 0)),
  ("Vacances d

#### Then we check if each row/date is a holiday

In [17]:
def is_holiday(date_str, zone):
    date_dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
    return any(start <= date_dt < end for _, start, end in holiday_ranges[zone])

In [18]:
def name_holiday(x):
    date_dt = datetime.strptime(x["Date"][:10], "%Y-%m-%d")

    current_zone = ""
    if x["Zone_A"] == True:
        current_zone = "Zone_A"
    elif x["Zone_B"] == True:
        current_zone = "Zone_B"
    elif x["Zone_C"] == True:
        current_zone = "Zone_C"
    else : 
        return x

    for holiday_name, start, end in holiday_ranges[current_zone]:
        if start <= date_dt < end:
            x[holiday_name] = 1
            break

    return x

In [19]:
for zone in zones:
    new_holidays[zone] = new_holidays["Date"].apply(lambda d : is_holiday(d, zone))

In [20]:
new_holidays

,Date,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,True,True,True,0,0,0,0,0
2,2021-01-01,True,True,True,0,0,0,0,0
4,2021-01-01,True,True,True,0,0,0,0,0
6,2021-01-01,True,True,True,0,0,0,0,0
8,2021-01-01,True,True,True,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,False,False,False,0,0,0,0,0
140251,2024-12-31,False,False,False,0,0,0,0,0
140253,2024-12-31,False,False,False,0,0,0,0,0
140255,2024-12-31,False,False,False,0,0,0,0,0


#### Now we add the information about the period of the holiday

In [21]:
new_holidays = new_holidays.apply(lambda x: name_holiday(x), axis=1)

In [22]:
new_holidays

,Date,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,True,True,True,0,1,0,0,0
2,2021-01-01,True,True,True,0,1,0,0,0
4,2021-01-01,True,True,True,0,1,0,0,0
6,2021-01-01,True,True,True,0,1,0,0,0
8,2021-01-01,True,True,True,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,False,False,False,0,0,0,0,0
140251,2024-12-31,False,False,False,0,0,0,0,0
140253,2024-12-31,False,False,False,0,0,0,0,0
140255,2024-12-31,False,False,False,0,0,0,0,0


## School holidays (py pipeline)

In [13]:
holidays2 = rp.school_holidays_preprocess(conso2, PATH_HOLIDAYS="data/calendar/", PATH_ARTIFACTS="artifacts/py_artifacts/")
holidays2

(Dictionnary successfully loaded)


,Date,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,True,True,True,0,1,0,0,0
2,2021-01-01,True,True,True,0,1,0,0,0
4,2021-01-01,True,True,True,0,1,0,0,0
6,2021-01-01,True,True,True,0,1,0,0,0
8,2021-01-01,True,True,True,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,False,False,False,0,0,0,0,0
140251,2024-12-31,False,False,False,0,0,0,0,0
140253,2024-12-31,False,False,False,0,0,0,0,0
140255,2024-12-31,False,False,False,0,0,0,0,0


## Asserts (school holidays)
**We want to see if the holidays, for each zone, respects the real dates of their respective holidays**  
We have tuples ("holiday", "start", "end").  
So we'll try to observe if our supposed school vacation dates actually fall between "start" and "end", by checking the boolean value in the same row from the column "zone".  
We check if the value is set to "True" during school holidays (for the current zone+year), and False when there are no holidays.

In [23]:
# We store the holidays for each zone in a dictionary
# It's a totally different thing, no we have dictionaries inside the dictionary dict_zones
dict_zones = {}

for zone in zones:

    # We select the date of the new dataset to facilitate the automation of the project in the future
    min_year = int(new_holidays["Date"].iloc[0][:4])
    max_year = int(new_holidays["Date"].iloc[-1][:4])

    dict_years = {}
    for date in range(min_year, max_year+1):
        ranges = []
        dict_years[date] = ranges
        for holiday in holiday_names:
            # The holidays "Vacances de Noël" start at "date-1" and end at "date".
            if holiday == "Vacances de Noël":
                holidays_filtered = holidays[
                                    (holidays["Zones"] == zone) & 
                                    (holidays["Description"] == holiday) &
                                    (holidays["annee_scolaire"] == f"{date-1}-{date}")
                                ]
            else : 
                holidays_filtered = holidays[
                                        (holidays["Zones"] == zone) & 
                                        (holidays["Description"] == holiday) &
                                        (holidays["annee_scolaire"] == f"{date}-{date+1}")
                                    ]
    
            start = datetime.strptime(holidays_filtered["Date de début"].iloc[0][:10], "%Y-%m-%d")
            end = datetime.strptime(holidays_filtered["Date de fin"].iloc[0][:10], "%Y-%m-%d")
            dict_years[date].append((holiday, start, end))
    
    dict_zones[zone] = dict_years

dict_zones

{'Zone_A': {2021: [('Vacances de la Toussaint',
    datetime.datetime(2021, 10, 22, 0, 0),
    datetime.datetime(2021, 11, 7, 0, 0)),
   ('Vacances de Noël',
    datetime.datetime(2020, 12, 18, 0, 0),
    datetime.datetime(2021, 1, 3, 0, 0)),
   ("Vacances d'Hiver",
    datetime.datetime(2022, 2, 11, 0, 0),
    datetime.datetime(2022, 2, 27, 0, 0)),
   ('Vacances de Printemps',
    datetime.datetime(2022, 4, 15, 0, 0),
    datetime.datetime(2022, 5, 1, 0, 0)),
   ("Vacances d'Été",
    datetime.datetime(2022, 7, 6, 0, 0),
    datetime.datetime(2022, 8, 31, 0, 0))],
  2022: [('Vacances de la Toussaint',
    datetime.datetime(2022, 10, 21, 0, 0),
    datetime.datetime(2022, 11, 6, 0, 0)),
   ('Vacances de Noël',
    datetime.datetime(2021, 12, 17, 0, 0),
    datetime.datetime(2022, 1, 2, 0, 0)),
   ("Vacances d'Hiver",
    datetime.datetime(2023, 2, 3, 0, 0),
    datetime.datetime(2023, 2, 19, 0, 0)),
   ('Vacances de Printemps',
    datetime.datetime(2023, 4, 7, 0, 0),
    datetime.date

In [24]:
new_holidays["Date"] = pd.to_datetime(new_holidays["Date"])
new_holidays

,Date,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,True,True,True,0,1,0,0,0
2,2021-01-01,True,True,True,0,1,0,0,0
4,2021-01-01,True,True,True,0,1,0,0,0
6,2021-01-01,True,True,True,0,1,0,0,0
8,2021-01-01,True,True,True,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,False,False,False,0,0,0,0,0
140251,2024-12-31,False,False,False,0,0,0,0,0
140253,2024-12-31,False,False,False,0,0,0,0,0
140255,2024-12-31,False,False,False,0,0,0,0,0


#### Pandas / Merge 2 datasets
doc : https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html  


**how : {‘left’, ‘right’, ‘outer’, ‘inner’, ‘cross’, ‘left_anti’, ‘right_anti’}**

*default ‘inner’ Type of merge to be performed.*

- left: use only keys from left frame, similar to a SQL left outer join; preserve key order.

- right: use only keys from right frame, similar to a SQL right outer join; preserve key order.

- outer: use union of keys from both frames, similar to a SQL full outer join; sort keys lexicographically.

- inner: use intersection of keys from both frames, similar to a SQL inner join; 
preserve the order of the left keys.

- cross: creates the cartesian product from both frames, preserves the order of the left keys.

- left_anti: use only keys from left frame that are not in right frame, similar to SQL left anti join; preserve key order.


**indicator : bool or str, default False**  
If True, adds a column to the output DataFrame called “_merge” with information on the source of each row. The column can be given a different name by providing a string argument. **The column will have a Categorical type with the value of “left_only” for observations whose merge key only appears in the left DataFrame**, “right_only” for observations whose merge key only appears in the right DataFrame, and “both” if the observation’s merge key is found in both DataFrames.


In [25]:
def check_if_correct(df, dict_zones):
    """
    For each zone and for each year, we filter the dataset by creating 2 subsets of the dataset. 
    1 dataset with containing all the dates of holidays (1). And Another one whithout the holidays (2).
    Then we check if the intersection of each date/row of (1) gives us the value 'True' 
    and we also check if the intersection of each row of (2) gives us the value 'False'.
    """
    for zone, year_dict in dict_zones.items():
        for year, holiday_list in year_dict.items():
            df_holidays = pd.DataFrame()
            df_no_holidays = pd.DataFrame()
            for name, start, end in holiday_list:
                
                with_holidays = df[ 
                    (start <= df["Date"]) & (df["Date"] < end)
                ]

                # We concatenate the holidays of an entire year to see if the intersection of the boolean value of each row gives us 
                # the value 'True'. 
                df_holidays = pd.concat([df_holidays, with_holidays], axis=0)
                
                
            # We construct the df without the days not in holidays
            
            df_no_holidays = df.merge(df_holidays, how="left", indicator=True)

            # The column will have a Categorical type with the value of “left_only” 
            # for observations whose merge key only appears in the left DataFrame
            df_no_holidays = df_no_holidays[ df_no_holidays['_merge'] == 'left_only' ].drop(columns=['_merge'])

            # We check if the dataset containing the days that are holidays only has the value True
            # for the the current zone + year
            # If an issue occurs with the dates, we can't tell which holiday name the issue comes from.
            # We can only know the zone and the year that causes a problem
            
            print(f"(Zone : {zone}, Year : {year})")
            assert df_holidays[zone].all(), f"df_holidays, issue"
            print("df_holidays good")

            # We check if the dataset, containing the days that are not holidays, only got the value False 
            # for the the current zone + year
            assert not df_no_holidays[zone].all(), "df_no_holidays issue"
            print("df_no_holidays good")

            print("\n")

            

In [26]:
check_if_correct(new_holidays, dict_zones)

(Zone : Zone_A, Year : 2021)
df_holidays good
df_no_holidays good


(Zone : Zone_A, Year : 2022)
df_holidays good
df_no_holidays good


(Zone : Zone_A, Year : 2023)
df_holidays good
df_no_holidays good


(Zone : Zone_A, Year : 2024)
df_holidays good
df_no_holidays good


(Zone : Zone_B, Year : 2021)
df_holidays good
df_no_holidays good


(Zone : Zone_B, Year : 2022)
df_holidays good
df_no_holidays good


(Zone : Zone_B, Year : 2023)
df_holidays good
df_no_holidays good


(Zone : Zone_B, Year : 2024)
df_holidays good
df_no_holidays good


(Zone : Zone_C, Year : 2021)
df_holidays good
df_no_holidays good


(Zone : Zone_C, Year : 2022)
df_holidays good
df_no_holidays good


(Zone : Zone_C, Year : 2023)
df_holidays good
df_no_holidays good


(Zone : Zone_C, Year : 2024)
df_holidays good
df_no_holidays good




## Merging of `conso` and `holidays2`

In [ ]:
# We merge them with pd.concat instead of pd.merge beacause the axis and the sizes are identical.
conso2 = pd.concat([conso2, holidays2.drop("Date", axis=1)], axis=1)

In [15]:
conso2.head()

,Date,Heures,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,00:00,67156.0,True,True,True,0,1,0,0,0
2,2021-01-01,00:30,66483.0,True,True,True,0,1,0,0,0
4,2021-01-01,01:00,64390.0,True,True,True,0,1,0,0,0
6,2021-01-01,01:30,64223.0,True,True,True,0,1,0,0,0
8,2021-01-01,02:00,63689.0,True,True,True,0,1,0,0,0


## Public holidays  
We're going to merge the dataset containing the public holidays (jours fériés) with the main dataset

In [28]:
feries = pd.read_csv("data/calendar/jours_feries_metropole.csv").drop(["annee", "zone"], axis=1) 

feries["date"] = feries["date"].apply(lambda x : datetime.strptime(x, "%Y-%m-%d"))
type(feries["date"].iloc[0])

pandas.Timestamp

In [29]:
feries = feries[
    (feries["date"] >= datetime.strptime(conso["Date"].iloc[0], "%Y-%m-%d")) &
    (feries["date"] <= datetime.strptime(conso["Date"].iloc[-1], "%Y-%m-%d"))
    ]
feries

,date,nom_jour_ferie
165,2021-01-01,1er janvier
166,2021-04-05,Lundi de Pâques
167,2021-05-01,1er mai
168,2021-05-08,8 mai
169,2021-05-13,Ascension
170,2021-05-24,Lundi de Pentecôte
171,2021-07-14,14 juillet
172,2021-08-15,Assomption
173,2021-11-01,Toussaint
174,2021-11-11,11 novembre


In [30]:
public_holidays = set(feries["date"])
public_holidays

{Timestamp('2021-01-01 00:00:00'),
 Timestamp('2021-04-05 00:00:00'),
 Timestamp('2021-05-01 00:00:00'),
 Timestamp('2021-05-08 00:00:00'),
 Timestamp('2021-05-13 00:00:00'),
 Timestamp('2021-05-24 00:00:00'),
 Timestamp('2021-07-14 00:00:00'),
 Timestamp('2021-08-15 00:00:00'),
 Timestamp('2021-11-01 00:00:00'),
 Timestamp('2021-11-11 00:00:00'),
 Timestamp('2021-12-25 00:00:00'),
 Timestamp('2022-01-01 00:00:00'),
 Timestamp('2022-04-18 00:00:00'),
 Timestamp('2022-05-01 00:00:00'),
 Timestamp('2022-05-08 00:00:00'),
 Timestamp('2022-05-26 00:00:00'),
 Timestamp('2022-06-06 00:00:00'),
 Timestamp('2022-07-14 00:00:00'),
 Timestamp('2022-08-15 00:00:00'),
 Timestamp('2022-11-01 00:00:00'),
 Timestamp('2022-11-11 00:00:00'),
 Timestamp('2022-12-25 00:00:00'),
 Timestamp('2023-01-01 00:00:00'),
 Timestamp('2023-04-10 00:00:00'),
 Timestamp('2023-05-01 00:00:00'),
 Timestamp('2023-05-08 00:00:00'),
 Timestamp('2023-05-18 00:00:00'),
 Timestamp('2023-05-29 00:00:00'),
 Timestamp('2023-07-

In [31]:
conso["public_holidays"] = pd.to_datetime(conso["Date"]).isin(public_holidays) + 0
conso.head()

,Date,Heures,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été,public_holidays
0,2021-01-01,00:00,67156.0,True,True,True,0,1,0,0,0,1
2,2021-01-01,00:30,66483.0,True,True,True,0,1,0,0,0,1
4,2021-01-01,01:00,64390.0,True,True,True,0,1,0,0,0,1
6,2021-01-01,01:30,64223.0,True,True,True,0,1,0,0,0,1
8,2021-01-01,02:00,63689.0,True,True,True,0,1,0,0,0,1


## Public Holidays (pipeline)

In [16]:
rp.public_holidays_preprocess(conso2, "data/calendar/")
conso2

,Date,Heures,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été,public_holidays
0,2021-01-01,00:00,67156.0,True,True,True,0,1,0,0,0,1
2,2021-01-01,00:30,66483.0,True,True,True,0,1,0,0,0,1
4,2021-01-01,01:00,64390.0,True,True,True,0,1,0,0,0,1
6,2021-01-01,01:30,64223.0,True,True,True,0,1,0,0,0,1
8,2021-01-01,02:00,63689.0,True,True,True,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,21:30,63946.0,False,False,False,0,0,0,0,0,0
140251,2024-12-31,22:00,63063.0,False,False,False,0,0,0,0,0,0
140253,2024-12-31,22:30,63816.0,False,False,False,0,0,0,0,0,0
140255,2024-12-31,23:00,65437.0,False,False,False,0,0,0,0,0,0


## Weather Dataset  
Our goal is to automate the cleaning of the weather datasets using python scripts.  
Source : https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-horaires

In [77]:
conso = None
if os.path.isfile("data/conso/conso.parquet"):
    conso = pd.read_parquet("data/conso/conso.parquet")
    print("conso dataset succesfully loaded")
else : 
    conso = rp.conso_preprocess("data/conso/")
    holidays2 = rp.school_holidays_preprocess(conso, PATH_HOLIDAYS="data/calendar/", PATH_ARTIFACTS="artifacts/py_artifacts/")
    conso = pd.concat([conso, holidays2.drop("Date", axis=1)], axis=1)
    rp.public_holidays_preprocess(conso, "data/calendar/")
    conso.to_parquet("data/conso/clean_conso/conso.parquet")
    print("conso dataset succesfully created")


conso dataset succesfully created


In [78]:
conso

,Date,Heures,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été,public_holidays
0,2021-01-01,00:00:00,67156.0,1,1,1,0,1,0,0,0,0
1,2021-01-01,00:30:00,66483.0,1,1,1,0,1,0,0,0,0
2,2021-01-01,01:00:00,64390.0,1,1,1,0,1,0,0,0,0
3,2021-01-01,01:30:00,64223.0,1,1,1,0,1,0,0,0,0
4,2021-01-01,02:00:00,63689.0,1,1,1,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
70123,2024-12-31,21:30:00,63946.0,0,0,0,0,0,0,0,0,0
70124,2024-12-31,22:00:00,63063.0,0,0,0,0,0,0,0,0,0
70125,2024-12-31,22:30:00,63816.0,0,0,0,0,0,0,0,0,0
70126,2024-12-31,23:00:00,65437.0,0,0,0,0,0,0,0,0,0


In [9]:
meteo75 = pd.read_csv("data/weather/H_75_previous-2020-2024.csv.gz", sep =';', compression="gzip")

In [10]:
meteo75

,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJHH,RR1,QRR1,DRR1,QDRR1,...,INS2,QINS2,TLAGON,QTLAGON,TVEGETAUX,QTVEGETAUX,ECOULEMENT,QECOULEMENT,STATUS_FXI3S,STATUS_DXI3S
0,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010100,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010101,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010102,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010103,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010104,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261855,75116008,LONGCHAMP,48.854833,2.233667,27,2024123119,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
261856,75116008,LONGCHAMP,48.854833,2.233667,27,2024123120,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
261857,75116008,LONGCHAMP,48.854833,2.233667,27,2024123121,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
261858,75116008,LONGCHAMP,48.854833,2.233667,27,2024123122,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0


In [11]:
cols_to_keep = [
    'NUM_POSTE',        # station ID : identifies the city
    'LAT', 'LON',       # coordinates: used to query Open-Meteo
    'AAAAMMJJHH',       # timestamp
    # Usefull features that we can found with the API of open-meteo
    'T',                # temperature_2m        
    'U',                # relative_humidity_2m  
    'FF',               # wind_speed_10m        
    'PMER',             # pressure_msl          
    'RR1',              # precipitation         
]

meteo75 = meteo75[cols_to_keep]

In [12]:
meteo75

,NUM_POSTE,LAT,LON,AAAAMMJJHH,T,U,FF,PMER,RR1
0,75106001,48.844833,2.338500,2020010100,1.6,NaN,NaN,NaN,0.0
1,75106001,48.844833,2.338500,2020010101,0.9,NaN,NaN,NaN,0.0
2,75106001,48.844833,2.338500,2020010102,0.1,NaN,NaN,NaN,0.0
3,75106001,48.844833,2.338500,2020010103,0.2,NaN,NaN,NaN,0.0
4,75106001,48.844833,2.338500,2020010104,0.6,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...
261855,75116008,48.854833,2.233667,2024123119,4.9,94.0,5.0,NaN,0.0
261856,75116008,48.854833,2.233667,2024123120,5.3,93.0,5.1,NaN,0.0
261857,75116008,48.854833,2.233667,2024123121,5.6,93.0,4.4,NaN,0.0
261858,75116008,48.854833,2.233667,2024123122,5.6,95.0,5.0,NaN,0.0


#### Let's separate the date and hours by creating 2 new columns and make them compatible with the main dataset `conso`

In [13]:
meteo75["Date"] = meteo75["AAAAMMJJHH"].apply(lambda x : str(str(x)[:8]))
meteo75["Date"] = meteo75["Date"].apply(lambda x : datetime.strptime(x, '%Y%m%d').date())

meteo75["Heures"] = meteo75["AAAAMMJJHH"].apply(lambda x : str(str(x)[8:]) + ':00')
meteo75["Heures"] = meteo75["Heures"].apply(lambda x : datetime.strptime(x, '%H:%M').time())

meteo75 = meteo75.drop("AAAAMMJJHH", axis=1)

In [14]:
meteo75.head(1)

,NUM_POSTE,LAT,LON,T,U,FF,PMER,RR1,Date,Heures
0,75106001,48.844833,2.3385,1.6,NaN,NaN,NaN,0.0,2020-01-01,00:00:00


Let's check if all the timeframes are present

In [15]:
meteo75[meteo75["NUM_POSTE"]==75106001]["Heures"].unique()

array([datetime.time(0, 0), datetime.time(1, 0), datetime.time(2, 0),
       datetime.time(3, 0), datetime.time(4, 0), datetime.time(5, 0),
       datetime.time(6, 0), datetime.time(7, 0), datetime.time(8, 0),
       datetime.time(9, 0), datetime.time(10, 0), datetime.time(11, 0),
       datetime.time(12, 0), datetime.time(13, 0), datetime.time(14, 0),
       datetime.time(15, 0), datetime.time(16, 0), datetime.time(17, 0),
       datetime.time(18, 0), datetime.time(19, 0), datetime.time(20, 0),
       datetime.time(21, 0), datetime.time(22, 0), datetime.time(23, 0)],
      dtype=object)

#### We'll keep the rows where the date is between `min(conso["Date"])` and `max(conso["Date"])`

In [16]:
min_date = conso["Date"].iloc[0]
max_date = conso["Date"].iloc[-1]

meteo75 = meteo75[
    (meteo75["Date"]>= min_date) & 
    (meteo75["Date"]<= max_date)
    ]

In [17]:
print(f"min : {meteo75["Date"].iloc[0]}", f"\nmax : {meteo75["Date"].iloc[-1]}")

min : 2021-01-01 
max : 2024-12-31


### We can observe that there are multiple stations / data providers.  
There are differents weather stations so there are several duplicate rows for a same date

In [15]:
rp.monitoring_nan(meteo75)

There are 6 stations
Percentage of NaN per column
NUM_POSTE     0.0
LAT           0.0
LON           0.0
T             0.2
U            66.5
FF           50.4
PMER         83.2
RR1          33.3
Date          0.0
Heures        0.0
dtype: float64

Percentage of NaN per station
                  T      U          FF   PMER         RR1
NUM_POSTE                                                
75106001   0.008556  100.0  100.000000  100.0    0.444901
75107005   1.182068  100.0    0.390082  100.0  100.000000
75110001   0.000000  100.0  100.000000  100.0    0.410678
75114001   0.000000    0.0    0.011408    0.0    0.002852
75114007   0.037076  100.0  100.000000  100.0  100.000000
75116008   0.077002    0.0    0.322268  100.0    1.026694


### We guess that some stations didn't record all the weather data, but that others did.  
Let's calculate the percentage of missing values for each columns and also grouped each station

We are going to keep the station that has the fewest missing values

In [18]:
meteo75 = rp.select_best_station(meteo75, conso)

The best station has a weighted average of :  0.02566735112936345
Best station :  75114001


In [17]:
rp.monitoring_nan(meteo75)

There are 1 stations
Percentage of NaN per column
NUM_POSTE    0.0
LAT          0.0
LON          0.0
T            0.0
U            0.0
FF           0.0
PMER         0.0
RR1          0.0
Date         0.0
Heures       0.0
dtype: float64

Percentage of NaN per station
             T    U        FF  PMER       RR1
NUM_POSTE                                    
75114001   0.0  0.0  0.011408   0.0  0.002852


### Now let's add the half-hour periods to the dataset

In [19]:
meteo75 = rp.half_hour(meteo75, conso)

In [20]:
meteo75

,Date,Heures,NUM_POSTE,LAT,LON,T,U,FF,PMER,RR1
0,2021-01-01,00:00:00,75114001,48.821667,2.337833,1.6,84.0,2.0,1008.0,0.0
1,2021-01-01,00:30:00,75114001,48.821667,2.337833,NaN,NaN,NaN,NaN,NaN
2,2021-01-01,01:00:00,75114001,48.821667,2.337833,1.4,84.0,1.8,1008.2,0.0
3,2021-01-01,01:30:00,75114001,48.821667,2.337833,NaN,NaN,NaN,NaN,NaN
4,2021-01-01,02:00:00,75114001,48.821667,2.337833,0.9,86.0,1.6,1008.6,0.0
...,...,...,...,...,...,...,...,...,...,...
70123,2024-12-31,21:30:00,75114001,48.821667,2.337833,NaN,NaN,NaN,NaN,NaN
70124,2024-12-31,22:00:00,75114001,48.821667,2.337833,5.2,93.0,6.9,1028.3,0.0
70125,2024-12-31,22:30:00,75114001,48.821667,2.337833,NaN,NaN,NaN,NaN,NaN
70126,2024-12-31,23:00:00,75114001,48.821667,2.337833,5.2,95.0,4.2,1028.0,0.0


#### For the missing values we'll use the **linear interpolation** method.
#### We could've used a **Gaussian Process** but it has a **too high computational complexity** for large datasets. The Gaussian Process would've been better for **long sequences of consecutive missing values**. Hence we'll keep the linear interpolation.
The interpolation method estimates missing values by fitting a function bewteen two known neighboring points and evaluating it at the missing timestamp (assuming the variable evolves smoothly and continuously bewteen two observations).  

We'll use the function below to estimate the missing value, like temperature, from a timestamp between the known values from its 2 neighboring timestamps:  

> $T(t) = T_{1} + (\frac{T_{2}-T_{1}}{t_{2}-t_{1}}) *  (t-t_{1})$

We calculate the slope $\alpha$ of the function that goes through two known points, $T_{1}$ and $T_{2}$, from two timestamps $t_{1}$ and $t_{2}$ as close together as possible :  
> $ \alpha = \frac{T_{2} - T_{1}}{t_{2}-t_{1}}$

We assume a linear relationship between these 2 points, so the affine function will look like this :  
> $f(t) = \alpha * t + \beta$,  
> with $ \beta = f(t) -\alpha*t$

We know the value $T_{1} = f(t_{1})$ for a surrounding timestamp $t_{1}$.  
So : 
> $\beta = T_{1} -\alpha*t_{1}$  

We could've chosen the other surrounding timestamp $t_{2}$ and its value $T_{2}$ instead of $t_{1}$ and $T_{1}$.

Then we substitute $\beta$ with ($T_{1} -\alpha*t_{1}$) : 
> $f(t) = \alpha * t + (T_{1} -\alpha*t_{1})$  
> $f(t) = T_{1} + \alpha * (t - t_{1})$  
> $f(t) = T_{1} + (\frac{T_{2}-T_{1}}{t_{2}-t_{1}}) *  (t-t_{1})$


In [28]:
def interpolation(x, n1, n2):
    x1, X1 = n1
    x2, X2 = n2
    return X1 + (x-x1)*(X2-X1)/(x2-x1)

In [29]:
pd.isna(meteo75["FF"].iloc[0])

False

In [17]:
def time_to_float(t):
    """Convert datetime.time to float hours 
    (example :  14:30 -> 14.5)
    """
    return t.hour + t.minute / 60


def interpolation_col(df):
    df = df.reset_index(drop=True)
    columns = list(set(df.columns) - set(["NUM_POSTE", "LAT", "LON", "Date", "Heures"]))
    
    for col in columns:
        # If the column has Nan Values, we apply the interpolation method on it
        if (meteo75[col].isnull().mean() * 100) != 0.0 :
            for i in range(len(df)):
                if(pd.isna(df[col].iloc[i])):
                    # If the missing value is in the first row, we choose the value of the next row
                    if(i == 0):
                        df[i, col] = df[col].iloc[i+1]
                    # If the missing value is in the last row, we choose the value of the last row
                    elif i == len(df)-1 : 
                        df[i, col] = df[col].iloc[i-1]
                    else : 
                        t = time_to_float(df["Heures"].iloc[i])
                        t1, T1 = time_to_float(df["Heures"].iloc[i-1]), df[col].iloc[i-1]    
                        # We select the index of the next valid row (Not NaN value)
                        # first_valid_index returns an index, not a position
                        idx_next = df[col].iloc[i+1:].first_valid_index()
                        if idx_next is None:
                            df.loc[i, col] = T1
                            continue

                        t2, T2 = time_to_float(df["Heures"].loc[idx_next]), df[col].loc[idx_next]
                        
                        df.loc[i, col] = interpolation(t, (t1, T1), (t2, T2))
    return df
                    

In [18]:
meteo75 = rp.interpolation_col(meteo75)

#### Now the Nan values are filled

In [19]:
rp.monitoring_nan(meteo75)

There are 1 stations
Percentage of NaN per column
NUM_POSTE    0.0
LAT          0.0
LON          0.0
T            0.0
U            0.0
FF           0.0
PMER         0.0
RR1          0.0
Date         0.0
Heures       0.0
dtype: float64

Percentage of NaN per station
             T    U   FF  PMER  RR1
NUM_POSTE                          
75114001   0.0  0.0  0.0   0.0  0.0


### We'll use the pandas built-in method to interpolate the missing values  
Since the pandas methods are faster, we'll use them  
  
Each **weather variable** has its **own temporal dynamics**. The **temperature** doesn't evolve **the same way** as **humidity** or **rain**. So we'll **adjust the interpolation limit** *(Maximum number of consecutive NaNs to fill)* **for each column**.

- Temperature : evolves slowly, quasi-linear. limit = 6 hours
- Humidity : the same physical behavior as the temperature.
- Wind : highly variable. limit = 6 hours
- Pressure : evolves slowly. limit = 12 hours
- Rain : the rain is unpredictable. It's better to assign the value 0 for the missing values than invent an incorrect scenario in which it's raining when it's not.

*We'll multiply by 2 the limit because we have data for every half-hour*

In [53]:
rp.interpolate_pd(meteo75)

In [54]:
meteo75

,Date,Heures,NUM_POSTE,LAT,LON,T,U,FF,PMER,RR1
0,2021-01-01,00:00:00,75114001,48.821667,2.337833,1.60,84.0,2.0,1008.00,0.0
1,2021-01-01,00:30:00,75114001,48.821667,2.337833,1.50,84.0,2.9,1008.10,0.0
2,2021-01-01,01:00:00,75114001,48.821667,2.337833,1.40,84.0,1.8,1008.20,0.0
3,2021-01-01,01:30:00,75114001,48.821667,2.337833,1.15,85.0,2.9,1008.40,0.0
4,2021-01-01,02:00:00,75114001,48.821667,2.337833,0.90,86.0,1.6,1008.60,0.0
...,...,...,...,...,...,...,...,...,...,...
70123,2024-12-31,21:30:00,75114001,48.821667,2.337833,5.15,93.0,2.9,1028.25,0.0
70124,2024-12-31,22:00:00,75114001,48.821667,2.337833,5.20,93.0,6.9,1028.30,0.0
70125,2024-12-31,22:30:00,75114001,48.821667,2.337833,5.20,94.0,2.9,1028.15,0.0
70126,2024-12-31,23:00:00,75114001,48.821667,2.337833,5.20,95.0,4.2,1028.00,0.0


In [55]:
rp.monitoring_nan(meteo75)

There are 1 stations
Percentage of NaN per column
Date         0.0
Heures       0.0
NUM_POSTE    0.0
LAT          0.0
LON          0.0
T            0.0
U            0.0
FF           0.0
PMER         0.0
RR1          0.0
dtype: float64

Percentage of NaN per station
             T    U   FF  PMER  RR1
NUM_POSTE                          
75114001   0.0  0.0  0.0   0.0  0.0


### Now let's clean and merge all the weather dataset from the different French departments into a single one using only our function `weather_clean_all()`
We'll create three different datasets. 
- A datasets that contains the temperature of the different departments of the 5 most crowded French regions (Ile-de-France, Auvergne-Rhône-Alpes,Hauts-de-France, Nouvelle-Aquitaine, Occitanie) + the weather data weighted by the population.
- A dataset that contains 5 temperature columns (from main departments) + other weather data weighted by population.
- A dataset that contains all weather data weighted by population.  

Then we'll train briefly few models on each dataset to keep the best one.


This function will be used inside the final function that will merge together all the datasets.

In [32]:
rp.weather_clean_all(conso, "data/weather", "data/weather/clean_weather")

Loading the file of the department n° 13 ...
File loaded with success
Adding the Date and Hours
Before selecting the best station for department n°13

Nb of stations :  23
Percentage of NaN per column :
NUM_POSTE     0.0
LAT           0.0
LON           0.0
T             5.1
U            30.1
FF           20.5
PMER         85.0
RR1           5.2
Date          0.0
Heures        0.0
dtype: float64

Percentage of NaN per station :
                    T           U          FF        PMER         RR1
NUM_POSTE                                                            
13001009     0.002852    0.002852    5.635410  100.000000    0.034223
13004003     0.000000    0.000000    0.048484  100.000000    0.082708
13005003     0.002852    0.000000    0.017113  100.000000    0.005704
13022003     0.000000    0.008556    0.014260  100.000000    0.002852
13028001     0.017112  100.000000    0.045631  100.000000    0.031371
13030001     0.014272  100.000000  100.000000  100.000000    0.219793
13031002 

### Testing the functions that create the final datasets ready for the training

In [79]:
rp.dataset_v1(conso, "data/weather/clean_weather", "data/final_datasets/")

In [80]:
df_test = pd.read_parquet("data/final_datasets/conso_v1.parquet")

In [81]:
df_test

,Date,Heures,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,...,13T,35T,67T,45T,76T,33T,U,FF,PMER,RR1
0,2021-01-01,00:00:00,67156.0,1,1,1,0,1,0,0,...,7.60,-0.30,2.10,-1.80,-0.10,3.40,7.709678,0.181087,83.959236,0.030442
1,2021-01-01,00:30:00,66483.0,1,1,1,0,1,0,0,...,7.80,-0.05,2.15,-1.80,0.45,3.70,7.733725,0.286171,83.962458,0.000000
2,2021-01-01,01:00:00,64390.0,1,1,1,0,1,0,0,...,8.00,0.20,2.20,-1.80,1.00,4.00,7.757772,0.212619,83.965679,0.016057
3,2021-01-01,01:30:00,64223.0,1,1,1,0,1,0,0,...,8.25,-0.50,2.50,-1.35,1.30,3.90,7.716159,0.286171,83.977223,0.000000
4,2021-01-01,02:00:00,63689.0,1,1,1,0,1,0,0,...,8.50,-1.20,2.80,-0.90,1.60,3.80,7.674546,0.177549,83.988767,0.006828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70123,2024-12-31,21:30:00,63946.0,0,0,0,0,0,0,0,...,5.95,8.85,-1.90,2.90,8.65,0.55,7.777459,0.286171,85.770976,0.000000
70124,2024-12-31,22:00:00,63063.0,0,0,0,0,0,0,0,...,5.70,8.70,-2.00,2.70,8.50,0.40,7.767843,0.409756,85.754795,0.000000
70125,2024-12-31,22:30:00,63816.0,0,0,0,0,0,0,0,...,5.50,8.30,-1.90,2.70,8.35,0.30,7.799066,0.286171,85.744596,0.000000
70126,2024-12-31,23:00:00,65437.0,0,0,0,0,0,0,0,...,5.30,7.90,-1.80,2.70,8.20,0.20,7.830289,0.384800,85.734396,0.000000


In [76]:
df_test.columns

Index(['Date', 'Heures', 'Consommation', 'Zone_A', 'Zone_B', 'Zone_C',
       'Vacances de la Toussaint', 'Vacances de Noël', 'Vacances d'Hiver',
       'Vacances de Printemps', 'Vacances d'Été', 'public_holidays', '44T',
       '69T', '31T', '21T', '59T', '75T', '13T', '35T', '67T', '45T', '76T',
       '33T', 'U', 'FF', 'PMER', 'RR1'],
      dtype='str')

### Testing area